In [1]:
import happybase
h = happybase.Connection("hbase", port=9090)
print("HBase tables:", h.tables())

HBase tables: []


In [2]:
%%writefile /home/jovyan/work/hbase_common.py
import os
from datetime import datetime, timezone
import happybase

HOST = os.environ.get("HBASE_HOST", "hbase")
SESSIONS = "user_sessions"
PRODUCT_DAILY = "product_daily"
MAX_TS = 9_999_999_999_999          # used to build a reverse timestamp (newest row sorts first)

def connect():
    return happybase.Connection(HOST, port=9090, timeout=120000)

def to_ms(iso):
    """ISO string (naive, treated as UTC) -> epoch milliseconds."""
    return int(datetime.fromisoformat(iso).replace(tzinfo=timezone.utc).timestamp() * 1000)

def rev_ts(iso):
    return f"{MAX_TS - to_ms(iso):013d}"

def session_key(user_id, start_time, session_id):
    """Row key: user_id # reverse_timestamp # session_id
    - same user's sessions are contiguous (prefix scan = one user)
    - reverse timestamp puts the newest session first
    - session_id keeps the key unique if two sessions start in the same millisecond."""
    return f"{user_id}#{rev_ts(start_time)}#{session_id}".encode()

def product_day_key(product_id, day):
    """Row key: product_id # yyyymmdd  (day like '2026-08-14')."""
    return f"{product_id}#{day.replace('-', '')}".encode()

def recreate_tables(h):
    existing = set(h.tables())
    for name, families in [
        (SESSIONS, {"meta": dict(max_versions=1),    # session info, device, geo, conversion
                    "pv":   dict(max_versions=1),    # one column per page view (sparse, variable)
                    "cart": dict(max_versions=1)}),  # one column per cart product (sparse)
        (PRODUCT_DAILY, {"stats": dict(max_versions=1)}),
    ]:
        if name.encode() in existing:
            h.delete_table(name, disable=True)
        h.create_table(name, families)
    return h.tables()

Writing /home/jovyan/work/hbase_common.py


In [3]:
import sys
sys.path.insert(0, "/home/jovyan/work")
import hbase_common as hc

h = hc.connect()
print("tables after create:", hc.recreate_tables(h))
t = h.table(hc.SESSIONS)

rows = [("user_000042", "2025-03-12T14:37:22", "sess_a7b3c9d8e2", "converted"),
        ("user_000042", "2025-03-14T09:10:00", "sess_bbbb000001", "browsed"),
        ("user_000042", "2025-03-10T20:00:00", "sess_cccc000002", "abandoned"),
        ("user_000173", "2025-03-11T08:00:00", "sess_dddd000003", "browsed")]
for u, st, sid, status in rows:
    t.put(hc.session_key(u, st, sid), {
        b"meta:session_id": sid.encode(), b"meta:start_time": st.encode(),
        b"meta:status": status.encode(),
        b"pv:00": b"home|||42", b"pv:01": b"product_detail|prod_00123|cat_007|124"})

print("\nsessions of user_000042, newest first:")
for key, data in t.scan(row_prefix=b"user_000042#", columns=[b"meta"]):
    print(" ", key.decode(), data[b"meta:start_time"].decode(), data[b"meta:status"].decode())

lo = hc.session_key("user_000042", "2025-03-13T00:00:00", "")   # newest bound
hi = hc.session_key("user_000042", "2025-03-11T00:00:00", "")   # oldest bound
print("\nrange 2025-03-11 .. 2025-03-13 for user_000042:")
for key, data in t.scan(row_start=lo, row_stop=hi, columns=[b"meta"]):
    print(" ", data[b"meta:start_time"].decode())

print("\nmost recent session, all families:")
for key, data in t.scan(row_prefix=b"user_000042#", limit=1):
    print(" ", key.decode(), {k.decode(): v.decode() for k, v in data.items()})

tables after create: [b'product_daily', b'user_sessions']

sessions of user_000042, newest first:
  user_000042#8258056599999#sess_bbbb000001 2025-03-14T09:10:00 browsed
  user_000042#8258209757999#sess_a7b3c9d8e2 2025-03-12T14:37:22 converted
  user_000042#8258363199999#sess_cccc000002 2025-03-10T20:00:00 abandoned

range 2025-03-11 .. 2025-03-13 for user_000042:
  2025-03-12T14:37:22

most recent session, all families:
  user_000042#8258056599999#sess_bbbb000001 {'meta:session_id': 'sess_bbbb000001', 'meta:start_time': '2025-03-14T09:10:00', 'meta:status': 'browsed', 'pv:00': 'home|||42', 'pv:01': 'product_detail|prod_00123|cat_007|124'}


In [4]:
%%writefile /home/jovyan/work/load_hbase.py
import json, os, sys, time, resource
from collections import defaultdict
sys.path.insert(0, "/home/jovyan/work")
import hbase_common as hc

D = sys.argv[1] if len(sys.argv) > 1 else "/home/jovyan/work/data"
BATCH = 1000

def load(name):
    with open(os.path.join(D, name)) as f:
        return json.load(f)

def s(v):
    return b"" if v is None else str(v).encode()

h = hc.connect()
print("tables:", hc.recreate_tables(h))
sessions_t = h.table(hc.SESSIONS)

# product_daily counters, filled while the sessions stream through
views = defaultdict(int)       # (product, day) -> product_detail page views
cart_adds = defaultdict(int)   # (product, day) -> sessions with the product in the cart

t0 = time.time()
n_rows = n_cells = 0
session_files = sorted(f for f in os.listdir(D) if f.startswith("sessions_") and f.endswith(".json"))
for fname in session_files:
    sessions = load(fname)                      # one file at a time keeps memory bounded
    with sessions_t.batch(batch_size=BATCH) as b:
        for x in sessions:
            geo, dev = x["geo_data"], x["device_profile"]
            day = x["start_time"][:10]
            cells = {
                b"meta:session_id": s(x["session_id"]), b"meta:user_id": s(x["user_id"]),
                b"meta:start_time": s(x["start_time"]), b"meta:end_time": s(x["end_time"]),
                b"meta:duration_s": s(x["duration_seconds"]), b"meta:status": s(x["conversion_status"]),
                b"meta:referrer": s(x["referrer"]), b"meta:device": s(dev["type"]),
                b"meta:os": s(dev["os"]), b"meta:browser": s(dev["browser"]),
                b"meta:city": s(geo["city"]), b"meta:state": s(geo["state"]),
                b"meta:country": s(geo["country"]), b"meta:ip": s(geo["ip_address"]),
            }
            for i, pv in enumerate(x["page_views"]):
                cells[f"pv:{i:02d}".encode()] = "|".join(
                    [pv["timestamp"], pv["page_type"], pv["product_id"] or "",
                     pv["category_id"] or "", str(pv["view_duration"])]).encode()
                if pv["page_type"] == "product_detail" and pv["product_id"]:
                    views[(pv["product_id"], day)] += 1
            for pid, c in x["cart_contents"].items():
                cells[f"cart:{pid}".encode()] = f"{c['quantity']}|{c['price']}".encode()
                cart_adds[(pid, day)] += 1
            b.put(hc.session_key(x["user_id"], x["start_time"], x["session_id"]), cells)
            n_rows += 1
            n_cells += len(cells)
            if n_rows % 20000 == 0:
                print(f"  {n_rows:,} sessions loaded ({time.time() - t0:.0f}s)")
    del sessions
print(f"user_sessions: {n_rows:,} rows, {n_cells:,} cells in {time.time() - t0:.0f}s")

# purchases per product per day come from the transactions file
units = defaultdict(int); units_linked = defaultdict(int); revenue = defaultdict(float)
for t in load("transactions.json"):
    day = t["timestamp"][:10]
    for it in t["items"]:
        k = (it["product_id"], day)
        units[k] += it["quantity"]
        revenue[k] += it["subtotal"]
        if t["session_id"]:
            units_linked[k] += it["quantity"]

keys = set(views) | set(cart_adds) | set(units)
pd_t = h.table(hc.PRODUCT_DAILY)
with pd_t.batch(batch_size=BATCH) as b:
    for (pid, day) in keys:
        b.put(hc.product_day_key(pid, day), {
            b"stats:views": str(views.get((pid, day), 0)).encode(),
            b"stats:cart_adds": str(cart_adds.get((pid, day), 0)).encode(),
            b"stats:units_sold": str(units.get((pid, day), 0)).encode(),
            b"stats:units_sold_linked": str(units_linked.get((pid, day), 0)).encode(),
            b"stats:revenue": f"{revenue.get((pid, day), 0.0):.2f}".encode()})
print(f"product_daily: {len(keys):,} rows")
print(f"total time {time.time() - t0:.0f}s | peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss // 1024} MB")

Writing /home/jovyan/work/load_hbase.py


In [5]:
!python /home/jovyan/work/load_hbase.py

tables: [b'product_daily', b'user_sessions']
  20,000 sessions loaded (22s)
  40,000 sessions loaded (29s)
  60,000 sessions loaded (34s)
  80,000 sessions loaded (40s)
  100,000 sessions loaded (47s)
  120,000 sessions loaded (73s)
  140,000 sessions loaded (80s)
  160,000 sessions loaded (85s)
  180,000 sessions loaded (91s)
  200,000 sessions loaded (97s)
user_sessions: 200,000 rows, 5,004,787 cells in 97s
product_daily: 369,425 rows
total time 128s | peak memory 1025 MB


In [6]:
%%writefile /home/jovyan/work/hbase_queries.py
import sys, time
sys.path.insert(0, "/home/jovyan/work")
import hbase_common as hc

h = hc.connect()
sess, daily = h.table(hc.SESSIONS), h.table(hc.PRODUCT_DAILY)
USER = "user_000042"

def timed(label, fn):
    t = time.time(); out = fn(); ms = (time.time() - t) * 1000
    print(f"[{ms:8.1f} ms] {label}"); return out

# Q1 - all sessions of one user, newest first (only the small 'meta' family is read)
rows = timed("Q1 sessions of one user (prefix scan, meta only)",
             lambda: list(sess.scan(row_prefix=f"{USER}#".encode(), columns=[b"meta"])))
print(f"   {len(rows)} sessions; newest 3:")
for key, d in rows[:3]:
    print("  ", d[b"meta:start_time"].decode(), d[b"meta:status"].decode(), d[b"meta:device"].decode(), d[b"meta:referrer"].decode())

# Q2 - the most recent session with all its page views (all families)
key, d = timed("Q2 latest session, all families",
               lambda: next(iter(sess.scan(row_prefix=f"{USER}#".encode(), limit=1))))
print("   session", d[b"meta:session_id"].decode(), "| status", d[b"meta:status"].decode())
for q in sorted(k for k in d if k.startswith(b"pv:")):
    ts, page, prod, cat, dur = d[q].decode().split("|")
    print(f"   {q.decode()}  {ts[11:]}  {page:16s} {prod:11s} {dur}s")
print("   cart:", {k.decode(): v.decode() for k, v in d.items() if k.startswith(b"cart:")})

# Q3 - time-range scan for one user: sessions in the last 30 days of the dataset
lo = hc.session_key(USER, "2026-11-05T00:00:00", ""); hi = hc.session_key(USER, "2026-10-05T00:00:00", "")
r = timed("Q3 user sessions 2026-10-05..2026-11-05 (row range)",
          lambda: list(sess.scan(row_start=lo, row_stop=hi, columns=[b"meta:start_time"])))
print(f"   {len(r)} sessions")

# Q4 - product performance over time: range scan on product_id#date, then conversion rates
PID = "prod_00123"
r = timed("Q4 product_daily range scan for one product, September",
          lambda: list(daily.scan(row_start=hc.product_day_key(PID, "2026-09-01"),
                                  row_stop=hc.product_day_key(PID, "2026-10-01"))))
v = sum(int(d[b"stats:views"]) for _, d in r); c = sum(int(d[b"stats:cart_adds"]) for _, d in r)
u = sum(int(d[b"stats:units_sold"]) for _, d in r)
ul = sum(int(d[b"stats:units_sold_linked"]) for _, d in r)
print(f"   {PID}: {len(r)} days | views {v} | cart adds {c} | units sold {u} (from sessions: {ul}) | view->cart {100*c/max(v,1):.1f}%")

# Q5 - why the row key matters: same question answered by scanning the WHOLE table with a filter
r = timed("Q5 same user via full-table RowFilter (no key design)",
          lambda: list(sess.scan(filter=f"RowFilter(=, 'substring:{USER}#')", columns=[b"meta:session_id"])))
print(f"   {len(r)} sessions found (same answer, scanned all rows)")

# Table sizes
n = timed("Q6 count user_sessions rows (keys only)",
          lambda: sum(1 for _ in sess.scan(filter="KeyOnlyFilter() AND FirstKeyOnlyFilter()")))
print(f"   user_sessions rows: {n:,}")

Writing /home/jovyan/work/hbase_queries.py


In [7]:
!python /home/jovyan/work/hbase_queries.py

[   226.8 ms] Q1 sessions of one user (prefix scan, meta only)
   19 sessions; newest 3:
   2026-11-02T11:38:44 abandoned tablet direct
   2026-10-31T08:34:36 abandoned tablet search_engine
   2026-10-28T12:41:40 browsed tablet email
[    75.3 ms] Q2 latest session, all families
   session sess_81ba89144d | status abandoned
   pv:00  11:38:44  search                       436s
   pv:01  11:46:00  search                       56s
   pv:02  11:46:56  search                       58s
   pv:03  11:47:54  category_listing             96s
   pv:04  11:49:30  product_detail   prod_03462  1033s
   pv:05  12:06:43  category_listing             87s
   pv:06  12:08:10  product_detail   prod_01415  45s
   pv:07  12:08:55  product_detail   prod_00499  271s
   cart: {'cart:prod_00499': '1|128.98'}
[    11.9 ms] Q3 user sessions 2026-10-05..2026-11-05 (row range)
   7 sessions
[    37.5 ms] Q4 product_daily range scan for one product, September
   prod_00123: 28 days | views 48 | cart adds 23 | units

In [8]:
%%writefile /home/jovyan/work/hbase_common.py
import os
from datetime import datetime, timezone
import happybase

HOST = os.environ.get("HBASE_HOST", "hbase")
SESSIONS = "user_sessions"
PRODUCT_DAILY = "product_daily"
# happybase leaves bloom filter and block cache OFF unless asked, so we turn them on explicitly
OPTS = dict(max_versions=1, bloom_filter_type="ROW", block_cache_enabled=True)
MAX_TS = 9_999_999_999_999          # used to build a reverse timestamp (newest row sorts first)

def connect():
    return happybase.Connection(HOST, port=9090, timeout=120000)

def to_ms(iso):
    """ISO string (naive, treated as UTC) -> epoch milliseconds."""
    return int(datetime.fromisoformat(iso).replace(tzinfo=timezone.utc).timestamp() * 1000)

def rev_ts(iso):
    return f"{MAX_TS - to_ms(iso):013d}"

def session_key(user_id, start_time, session_id):
    """Row key: user_id # reverse_timestamp # session_id
    - same user's sessions are contiguous (prefix scan = one user)
    - reverse timestamp puts the newest session first
    - session_id keeps the key unique if two sessions start in the same millisecond."""
    return f"{user_id}#{rev_ts(start_time)}#{session_id}".encode()

def product_day_key(product_id, day):
    """Row key: product_id # yyyymmdd  (day like '2026-08-14')."""
    return f"{product_id}#{day.replace('-', '')}".encode()

def recreate_tables(h):
    existing = set(h.tables())
    for name, families in [
        (SESSIONS, {"meta": dict(OPTS),    # session info, device, geo, conversion
                    "pv":   dict(OPTS),    # one column per page view (sparse, variable)
                    "cart": dict(OPTS)}),  # one column per cart product (sparse)
        (PRODUCT_DAILY, {"stats": dict(OPTS)}),
    ]:
        if name.encode() in existing:
            h.delete_table(name, disable=True)
        h.create_table(name, families)
    return h.tables()

Overwriting /home/jovyan/work/hbase_common.py


In [9]:
!python /home/jovyan/work/load_hbase.py

tables: [b'product_daily', b'user_sessions']
  20,000 sessions loaded (26s)
  40,000 sessions loaded (34s)
  60,000 sessions loaded (43s)
  80,000 sessions loaded (50s)
  100,000 sessions loaded (60s)
  120,000 sessions loaded (83s)
  140,000 sessions loaded (90s)
  160,000 sessions loaded (100s)
  180,000 sessions loaded (107s)
  200,000 sessions loaded (117s)
user_sessions: 200,000 rows, 5,004,787 cells in 118s
product_daily: 369,425 rows
total time 156s | peak memory 1025 MB


In [10]:
!python /home/jovyan/work/hbase_queries.py

[   210.5 ms] Q1 sessions of one user (prefix scan, meta only)
   19 sessions; newest 3:
   2026-11-02T11:38:44 abandoned tablet direct
   2026-10-31T08:34:36 abandoned tablet search_engine
   2026-10-28T12:41:40 browsed tablet email
[    10.6 ms] Q2 latest session, all families
   session sess_81ba89144d | status abandoned
   pv:00  11:38:44  search                       436s
   pv:01  11:46:00  search                       56s
   pv:02  11:46:56  search                       58s
   pv:03  11:47:54  category_listing             96s
   pv:04  11:49:30  product_detail   prod_03462  1033s
   pv:05  12:06:43  category_listing             87s
   pv:06  12:08:10  product_detail   prod_01415  45s
   pv:07  12:08:55  product_detail   prod_00499  271s
   cart: {'cart:prod_00499': '1|128.98'}
[     8.6 ms] Q3 user sessions 2026-10-05..2026-11-05 (row range)
   7 sessions
[    16.3 ms] Q4 product_daily range scan for one product, September
   prod_00123: 28 days | views 48 | cart adds 23 | units